# 04 · Build RAW / RAW-MATCHED / CURATED training sets
**Purpose:** RAW = every generated QA (passed + rejected); CURATED = passed the CuratorKIT gates; RAW-MATCHED = random sample of RAW with |CURATED| rows (seed 42) — the size control.
The same minimal format filter (non-empty question, parseable `Answer:` line) is applied to every arm.
**Inputs:** `artifacts/curator/full/chunk_*/{passed,rejected}.jsonl`. **Outputs:** `artifacts/train/{raw,raw_matched,curated}.jsonl` (messages format), `data/training_set_stats.json`.
**Runtime:** <1 min (CPU).

In [ ]:
import os, sys, json, random
from collections import Counter
if not os.path.ismount("/content/drive"):
    from google.colab import drive; drive.mount("/content/drive")
sys.path.insert(0, "/content/drive/MyDrive/ps8/repo/src")
import ps8_common as C
cfg = C.load_config(); FORCE = False
SEED = cfg["seed"]
FULL, OUT = C.ART / "curator" / "full", C.ART / "train"
OUT.mkdir(parents=True, exist_ok=True)
chunks = sorted(FULL.glob("chunk_*"))
assert chunks and all((d / "passed.jsonl").exists() for d in chunks), "run notebook 03 first"

In [ ]:
passed = [r for d in chunks for r in C.read_jsonl(d / "passed.jsonl")]
rejected = [r for d in chunks for r in C.read_jsonl(d / "rejected.jsonl") if r["source_pubid"] is not None]
print(f"passed={len(passed)} rejected QA={len(rejected)} (source-level generation failures excluded: they contain no QA)")

def well_formed(r):
    return bool(r["question"].strip()) and C.parse_answer(r["answer"]) is not None

test_ids = set(json.load(open(C.DATA / "test_pubids.json"))) | set(json.load(open(C.DATA / "dev_pubids.json")))
assert not ({int(r["source_pubid"]) for r in passed + rejected} & test_ids), "TEST/DEV leakage"

raw_all = passed + rejected
raw = [r for r in raw_all if well_formed(r)]
curated = [r for r in passed if well_formed(r)]
raw_matched = random.Random(SEED).sample(raw, len(curated))
dropped = {"raw": len(raw_all) - len(raw), "curated": len(passed) - len(curated)}
print("dropped by minimal format filter:", dropped)

In [ ]:
def reasoning_words(a):
    return len(a[:a.lower().rfind("answer:")].split())

sets = {"raw": raw, "raw_matched": raw_matched, "curated": curated}
stats = {"format_filter_dropped": dropped, "sets": {}}
for name, rows in sets.items():
    C.write_jsonl([{"messages": C.build_messages(r["context_text"], r["question"], r["answer"].strip())} for r in rows],
                  OUT / f"{name}.jsonl")
    labels = Counter(C.parse_answer(r["answer"]) for r in rows)
    stats["sets"][name] = {"n": len(rows), "labels": dict(labels),
                           "label_frac": {k: round(v / len(rows), 3) for k, v in labels.items()},
                           "avg_reasoning_words": round(sum(reasoning_words(r["answer"]) for r in rows) / len(rows), 1),
                           "n_source_abstracts": len({r["source_pubid"] for r in rows})}
stats["raw_rejection_breakdown"] = json.load(open(C.DATA / "curation_stats.json"))
json.dump(stats, open(C.DATA / "training_set_stats.json", "w"), indent=1)

import pandas as pd
print(pd.DataFrame(stats["sets"]).T[["n", "label_frac", "avg_reasoning_words", "n_source_abstracts"]].to_string())
print("Saved:", sorted(p.name for p in OUT.glob("*.jsonl")), C.DATA / "training_set_stats.json")